# RAG pipeline

This notebook implements the full RAG pipeline. It includes retrieving relevant chunks from the dataset, building context, and using the model to generate answers.

The goal of this notebook is to show how the RAG system works and how it uses the retrieved lecture content to answer questions.

At the end, I used the same 4 questions as the baseline model to test this RAG pipeline and verify that it works properly.

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

## Set OpenAI API key

In this step, I set the OpenAI API key and create the client, which will be used later to get embeddings and generate answers.

In [2]:
os.environ["OPENAI_API_KEY"] = "Your API" #You should replace this part with your own API.

client = OpenAI()

## Load chunk data

In this step, I load the ds593_chunks.csv dataset. This file contains the lecture materials that have been split into chunks, including information such as lecture name, source file, page number, chunk index, and text content.

In [6]:
df = pd.read_csv("ds593_chunks.csv")
df.head()

,chunk_id,lecture,source_file,page,chunk_index,text
0,L6_p1_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,1,1,Lecture 6 - Attention Mechanisms\nWelcome back...
1,L6_p2_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,1,The classic test\nAgenda for today\n1. Quick r...
2,L6_p2_c2,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,2,From Lecture 5:\nInput sequence -> Encoder -> ...
3,L6_p2_c3,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,3,Long inputs lose information:\nShort sentence ...
4,L6_p3_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,3,1,"Share with your neighbor, can they guess what ..."


In [7]:
# Check the column names of the files
df.columns

Index(['chunk_id', 'lecture', 'source_file', 'page', 'chunk_index', 'text'], dtype='object')

## Create embeddings for each chunk

In this step, I generate embeddings for each text chunk.

I then add the embeddings as a new column and save them to a new file called ds593_chunks_with_embeddings.pkl.

This file makes it easier to use these embeddings in later steps.

In [11]:
def get_embedding(text):
    response = client.embeddings.create(model="text-embedding-3-small", input=str(text))
    return response.data[0].embedding

In [10]:
# Load existing embeddings or create new ones
embedding_file = "ds593_chunks_with_embeddings.pkl"

if os.path.exists(embedding_file):
    df = pd.read_pickle(embedding_file)
    print("Loaded existing embeddings")
else:
    print("Creating embeddings (this may take a while)...")

    tqdm.pandas()
    df["embedding"] = df["text"].progress_apply(get_embedding)

    df.to_pickle(embedding_file)
    print("Embeddings created and saved")

Loaded existing embeddings


In [12]:
# Saved embeddings
df = pd.read_pickle("ds593_chunks_with_embeddings.pkl")
df.head()

,chunk_id,lecture,source_file,page,chunk_index,text,embedding
0,L6_p1_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,1,1,Lecture 6 - Attention Mechanisms\nWelcome back...,"[0.01058197021484375, 0.034942626953125, -0.04..."
1,L6_p2_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,1,The classic test\nAgenda for today\n1. Quick r...,"[0.0213775634765625, 0.0101776123046875, -0.01..."
2,L6_p2_c2,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,2,From Lecture 5:\nInput sequence -> Encoder -> ...,"[0.0247955322265625, -0.0084075927734375, -0.0..."
3,L6_p2_c3,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,2,3,Long inputs lose information:\nShort sentence ...,"[-0.0220794677734375, 0.01557159423828125, -0...."
4,L6_p3_c1,Lecture 6 - Attention Mechanisms,DS 593 Lecture 6.pdf,3,1,"Share with your neighbor, can they guess what ...","[0.0221405029296875, -0.01525115966796875, -0...."


## Build retrieval function

In this step, I build a function to retrieve the most relevant chunks for each question.

I compute the similarity between the question embedding and all chunk embeddings, and rank them based on the similarity scores.

Then I select the top k most similar chunks, which will be used later to build the context for the RAG system.

In [20]:
def retrieve_chunks(question, top_k=5):
    question_embedding = get_embedding(question)

    chunk_embeddings = np.array(df["embedding"].tolist())
    question_embedding = np.array(question_embedding).reshape(1, -1)

    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

    df_with_scores = df.copy()
    df_with_scores["similarity"] = similarities

    top_chunks = df_with_scores.sort_values("similarity", ascending=False).head(top_k)

    return top_chunks

## Test retrieval

In this step, I select 4 test questions that are the same as the ones used in the baseline model to test the retrieval function.

For each question, I perform retrieval to get the most relevant chunks.

Then, I select the top k results for each question and organize them into a retrieval results table.

In [21]:
questions = ["What is attention?",
             "What is the transformer?",
             "What is prompting?",
             "What is RAG?"]

all_results = []

for q in questions:
    top_chunks = retrieve_chunks(q, top_k=5)

    print("\nQuestion:", q)
    display(top_chunks[["lecture", "page", "text", "similarity"]])

    all_results.append({"question": q,
                        "top_lecture": top_chunks.iloc[0]["lecture"],
                        "top_page": top_chunks.iloc[0]["page"],
                        "top_similarity": top_chunks.iloc[0]["similarity"],
                        "top_text": top_chunks.iloc[0]["text"]})


Question: What is attention?


,lecture,page,text,similarity
10,Lecture 6 - Attention Mechanisms,4,This is exactly how attention works!\nAttentio...,0.573483
0,Lecture 6 - Attention Mechanisms,1,Lecture 6 - Attention Mechanisms\nWelcome back...,0.532670
7,Lecture 6 - Attention Mechanisms,3,Three roles in attention\nAttention uses three...,0.471280
99,Lecture 9 - Pre-training LLMs,1,"Recap: You've seen transformers\nIn Weeks 4-5,...",0.469744
41,Lecture 6 - Attention Mechanisms,16,Problem: A single attention mechanism tries to...,0.466516



Question: What is the transformer?


,lecture,page,text,similarity
71,Lecture 7 - Transformer Architecture,8,What surprised you?\nWhat seems clever?\nWhat ...,0.476749
100,Lecture 9 - Pre-training LLMs,2,Transformer architecture (encoder + decoder bl...,0.472136
50,Lecture 7 - Transformer Architecture,1,1. Recap + Data ﬂow: From text to Q/K/V\n2. Bu...,0.421869
49,Lecture 7 - Transformer Architecture,1,Lecture 7 - Transformer Architecture\nWelcome ...,0.399313
67,Lecture 7 - Transformer Architecture,7,"In transformers: EVERY sublayer (attention, FF...",0.363801



Question: What is prompting?


,lecture,page,text,similarity
164,Lecture 13 - Prompt Engineering and Prompt Inj...,1,API access is cheaper and faster than ﬁne-tuni...,0.564694
162,Lecture 13 - Prompt Engineering and Prompt Inj...,1,Lecture 13 - Prompt Engineering and\nPrompt In...,0.546252
163,Lecture 13 - Prompt Engineering and Prompt Inj...,1,1. Prompt engineering - techniques for getting...,0.538717
187,Lecture 13 - Prompt Engineering and Prompt Inj...,8,Shifting gears: Prompts as a security concern\...,0.536090
186,Lecture 13 - Prompt Engineering and Prompt Inj...,8,To clarify CoT vs. reasoning models:\nCoT is a...,0.490608



Question: What is RAG?


,lecture,page,text,similarity
270,Lecture 15 - Retrieval-Augmented Generation Pa...,16,Evaluating RAG systems\nLab due this week on RAG,0.764600
284,Lecture 16 - Building RAG Systems Part 2,5,Elements of a good RAG prompt:\nClear instruct...,0.613756
306,Lecture 16 - Building RAG Systems Part 2,12,Evaluating RAG: Two things can go wrong\nRetri...,0.558147
271,Lecture 16 - Building RAG Systems Part 2,1,Lecture 16 - Building RAG Systems (Part 2)\nIc...,0.557350
230,Lecture 15 - Retrieval-Augmented Generation Pa...,6,"RAG: Look up speciﬁc product info, policy deta...",0.531270


In [22]:
# Create a table for the top k of each problem
retrieval_results_df = pd.DataFrame(all_results)
retrieval_results_df

,question,top_lecture,top_page,top_similarity,top_text
0,What is attention?,Lecture 6 - Attention Mechanisms,4,0.573483,This is exactly how attention works!\nAttentio...
1,What is the transformer?,Lecture 7 - Transformer Architecture,8,0.476749,What surprised you?\nWhat seems clever?\nWhat ...
2,What is prompting?,Lecture 13 - Prompt Engineering and Prompt Inj...,1,0.564694,API access is cheaper and faster than ﬁne-tuni...
3,What is RAG?,Lecture 15 - Retrieval-Augmented Generation Pa...,16,0.764600,Evaluating RAG systems\nLab due this week on RAG


## Build RAG prompt

In this step, I first use the build_context function to combine the retrieved chunks into a context.

Then I combine the context, the question, and some instructions to build the prompt.

Finally, I send the prompt to the model to generate the RAG answer based on the given context.

In [23]:
def build_context(top_chunks):
    context = ""
    for i, row in top_chunks.iterrows():
        context += f"\n[Source: {row['lecture']} | Page {row['page']}]\n"
        context += row["text"]
        context += "\n"
    return context

In [24]:
# RAG answer
def generate_rag_answer(question, top_k=5):
    top_chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(top_chunks)

    prompt = f"""
              Use the provided context to answer the question as best as possible.

              If the answer is partially available, explain based on the context.
              Only say "not enough information" if absolutely necessary.

              Context:
              {context}

              Question:
              {question}

              Answer:
              """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You answer questions using only the provided course materials."},
                  {"role": "user", "content": prompt}],
                  temperature=0)

    answer = response.choices[0].message.content

    return answer, top_chunks

## Test RAG system

In this step, I use 4 questions to test the full RAG system.

For each question, I generate the RAG answer and display the retrieved chunks.

Then I save the results into a DataFrame to review the answers.

In [25]:
questions = ["What is attention?",
             "What is the transformer?",
             "What is prompting?",
             "What is RAG?"]

results = []

for q in questions:
    answer, sources = generate_rag_answer(q, top_k=5)

    print("\n==============================")
    print("Question:", q)

    print("\nRAG Answer:")
    print(answer)

    print("\nRetrieved Sources:")
    display(sources[["lecture", "page", "similarity", "text"]])

    results.append({"question": q, "rag_answer": answer})


Question: What is attention?

RAG Answer:
Attention is a mechanism used in natural language processing (NLP) that allows models to focus on different parts of the input data when generating output. It solves the bottleneck problem in encoder-decoder models by enabling the model to dynamically prioritize and attend to the most relevant parts of the input sequence. Attention is used in various applications beyond translation, such as document summarization and image captioning, where it helps the model focus on the most pertinent information or regions. The attention mechanism involves three roles: Query (Q), Key (K), and Value (V), which help determine what the model should focus on. Multi-head attention extends this concept by running multiple attention mechanisms in parallel, allowing each to learn to focus on different aspects of the input.

Retrieved Sources:


,lecture,page,similarity,text
10,Lecture 6 - Attention Mechanisms,4,0.573528,This is exactly how attention works!\nAttentio...
0,Lecture 6 - Attention Mechanisms,1,0.532716,Lecture 6 - Attention Mechanisms\nWelcome back...
7,Lecture 6 - Attention Mechanisms,3,0.471314,Three roles in attention\nAttention uses three...
99,Lecture 9 - Pre-training LLMs,1,0.469768,"Recap: You've seen transformers\nIn Weeks 4-5,..."
41,Lecture 6 - Attention Mechanisms,16,0.466540,Problem: A single attention mechanism tries to...



Question: What is the transformer?

RAG Answer:
The transformer is an architecture used in natural language processing tasks, characterized by its encoder-decoder structure. It is designed to handle sequence-to-sequence tasks, such as translation. The architecture includes key components like attention mechanisms, specifically self-attention and multi-head attention, as well as positional encoding, residual connections, layer normalization, and feed-forward networks (FFN). Each sublayer within the transformer, such as attention and FFN, incorporates residual connections and layer normalization to stabilize training. The transformer architecture is foundational to many large language models (LLMs) like GPT and BERT.

Retrieved Sources:


,lecture,page,similarity,text
71,Lecture 7 - Transformer Architecture,8,0.476749,What surprised you?\nWhat seems clever?\nWhat ...
100,Lecture 9 - Pre-training LLMs,2,0.472127,Transformer architecture (encoder + decoder bl...
50,Lecture 7 - Transformer Architecture,1,0.421906,1. Recap + Data ﬂow: From text to Q/K/V\n2. Bu...
49,Lecture 7 - Transformer Architecture,1,0.399328,Lecture 7 - Transformer Architecture\nWelcome ...
67,Lecture 7 - Transformer Architecture,7,0.363817,"In transformers: EVERY sublayer (attention, FF..."



Question: What is prompting?

RAG Answer:
Prompting is the process of changing the input to fit the model or task in order to get better outputs from language models. It involves crafting specific prompts to unlock capabilities of the model and optimize its performance without altering the model itself. Prompting is a key technique in interacting with language models, as it is more systematic and cost-effective compared to fine-tuning.

Retrieved Sources:


,lecture,page,similarity,text
164,Lecture 13 - Prompt Engineering and Prompt Inj...,1,0.564700,API access is cheaper and faster than ﬁne-tuni...
162,Lecture 13 - Prompt Engineering and Prompt Inj...,1,0.546195,Lecture 13 - Prompt Engineering and\nPrompt In...
163,Lecture 13 - Prompt Engineering and Prompt Inj...,1,0.538685,1. Prompt engineering - techniques for getting...
187,Lecture 13 - Prompt Engineering and Prompt Inj...,8,0.536031,Shifting gears: Prompts as a security concern\...
186,Lecture 13 - Prompt Engineering and Prompt Inj...,8,0.490598,To clarify CoT vs. reasoning models:\nCoT is a...



Question: What is RAG?

RAG Answer:
RAG, or Retrieval-Augmented Generation, is a system that combines retrieval and generation processes to provide answers or information. It involves an end-to-end pipeline where data is first chunked, embedded, and stored offline. Then, during the online phase, relevant chunks are retrieved, augmented, and used to generate responses. The system is designed to look up specific information, such as product details or policy information, by retrieving relevant chunks of data and generating answers based on them. The effectiveness of a RAG system depends on the chunking strategy, as it influences what the retriever can find and how well the system can generate accurate and contextually grounded responses.

Retrieved Sources:


,lecture,page,similarity,text
270,Lecture 15 - Retrieval-Augmented Generation Pa...,16,0.764600,Evaluating RAG systems\nLab due this week on RAG
284,Lecture 16 - Building RAG Systems Part 2,5,0.613756,Elements of a good RAG prompt:\nClear instruct...
306,Lecture 16 - Building RAG Systems Part 2,12,0.558147,Evaluating RAG: Two things can go wrong\nRetri...
271,Lecture 16 - Building RAG Systems Part 2,1,0.557350,Lecture 16 - Building RAG Systems (Part 2)\nIc...
230,Lecture 15 - Retrieval-Augmented Generation Pa...,6,0.531270,"RAG: Look up speciﬁc product info, policy deta..."


In [26]:
# Create RAG result table
rag_results_df = pd.DataFrame(results)
rag_results_df

,question,rag_answer
0,What is attention?,Attention is a mechanism used in natural langu...
1,What is the transformer?,The transformer is an architecture used in nat...
2,What is prompting?,Prompting is the process of changing the input...
3,What is RAG?,"RAG, or Retrieval-Augmented Generation, is a s..."


## Challenge

I encountered a challenge in this RAG_pipeline. I found that sometimes the retrieved context was not enough to fully answer the question. This caused the RAG generated answers to miss some important information. For example, when the question is “What is attention?”, the model could not find the information about the Value (V) component.

When I tried to find the reason, I found that this might be because I initially set top_k = 3, which limited the number of retrieved chunks and caused some important information to be missed. Since the content in my dataset is distributed across multiple chunks, retrieving only a few chunks is clearly not enough.

To solve this problem, I increased top_k to 5. After this change, the retrieved context contains more relevant information, and the generated RAG answers are more complete and cover most of the key points.